# DS6 — Statistical Mechanics (StM) Model

**Stage 4, Step 1 — Theoretical and Conceptual Modeling**
**Dataset:** DS6 (Danish Adolescents, quasi-experimental, N=243 behavioral)
**Model family:** Statistical Mechanics (StM)
**Date:** 2026-08-06
**Blueprint:** `outputs/reports/stage4/step1_modeling/ds6_stm_blueprint.md`

---

## Scope and purpose

Sixth and LAST active DS6 CAS model — closes all 7 (StoP, NeT, InT, DyS, SyD, StM; GaT/ABM
remain deferred pending Step 2, matching DS5/Structured precedent). Depends on InT (75) for
a sanity check only, not on StoP.

- **Goal:** Represent DS6 events as joint (app, time-of-day) microstate configurations;
  summarize per-participant macrostate structure (entropy, dominant configuration,
  persistence, susceptibility) per period.
- **Not goal:** Phenotype classification, personalization, cross-dataset harmonization.
- **Output:** Per-participant macrostate metrics, InT bridge sanity check, condition x
  period summary, psychosocial correlations.

### FLAG decisions in effect (blueprint Part 0-3)

| Flag | Decision |
|---|---|
| FLAG StM-1 | NO beta_behav/T_behav/Z/F thermodynamic apparatus -- DS6's unbounded StoP state variable makes DS5's beta=2/sigma^2 bridge even less applicable than it was for Structured, which already refused it |
| FLAG StM-2 | Microstate = joint (app_name, hour_bucket) -- paper's own named configuration properties (target app, time of day), hour buckets reuse Stage 1's existing morning/evening boundaries |
| FLAG StM-3 | H_joint >= max(H_app, H_hour) as a mandatory sanity gate (same discipline that caught 2 real bugs in the InT notebook) |
| FLAG StM-4 | Susceptibility = Jensen-Shannon divergence between baseline and intervention joint distributions (bounded, symmetric, support-robust -- not an invented metric) |

### WARNING — proxy status, and why no beta/T/Z/F here

Paper's own caution (§2.1): "physical quantities such as energy or temperature are not
directly observed... treated as behavioral analogies... not literal measurements." DS5's
`beta_behav = 2/sig_hat^2` bridge assumed a bounded [0,1] two-state occupation probability
-- DS6's StoP state variable (daily attempt count) has no upper bound, so even the loose
analogy breaks down more than it did for Structured (which already declined to build this
apparatus for its own, different reasons). This notebook does not build beta/T/Z/F.
Macrostate structure is described directly via entropy, dominant-configuration persistence,
and distributional-shift susceptibility -- all standard information-theoretic quantities,
not physics quantities dressed up in behavioral language.

---

## Section 1: Pure Theory — Statistical Mechanics in CAS (paper Section 2.1)

> "Properties such as condition, target app, time, prior attempts, and resolution code
> define the event-state configuration... a single opening attempt is represented as a
> configuration of behavioral features: the target app... the time of day... the current
> resolution."

DS6 cannot use "resolution code"/"prior outcome" (no outcome variance). Uses the paper's
other two named properties: **target app** and **time of day**.

### Core quantities (no beta/T/Z/F -- see WARNING above)

    Microstate:        s = (app_name, hour_bucket)
    Macrostate entropy: H_joint = -sum_s p(s) * log2(p(s))
    Dominant config:    argmax_s p(s)
    Persistence:        1[dominant_config_baseline == dominant_config_intervention]
    Susceptibility:     JS_divergence(P_baseline(s), P_intervention(s))
                       = 0.5*KL(P_base || M) + 0.5*KL(P_intv || M),  M = 0.5*(P_base + P_intv)

In [ ]:
# =============================================================================
# Imports and paths
# =============================================================================
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats
from scipy.spatial.distance import jensenshannon
from datetime import datetime

ROOT    = Path('..').resolve()
CLEANED = ROOT / 'outputs' / 'cleaned'
FEAT    = ROOT / 'outputs' / 'features' / 'ds6'
INT_DIR = ROOT / 'outputs' / 'model_outputs' / 'ds6' / 'information_theory'
REPORTS = ROOT / 'outputs' / 'model_outputs' / 'ds6' / 'statistical_mechanics'
FIGURES = REPORTS / 'figures'
REPORTS.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

MODEL_RUN_TS = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
print('REPORTS :', REPORTS)
print('Run at  :', MODEL_RUN_TS)

In [ ]:
# =============================================================================
# Load data, build hour_bucket (reuses Stage 1's morning/evening boundaries)
# =============================================================================
beh = pd.read_csv(CLEANED / 'ds6_behavioral_clean.csv', low_memory=False)
beh['event_timestamp_utc'] = pd.to_datetime(beh['event_timestamp_utc'], utc=True)

def hour_to_bucket(h):
    if 0 <= h <= 5:
        return 'night'
    elif 6 <= h <= 11:
        return 'morning'
    elif 12 <= h <= 17:
        return 'afternoon'
    else:
        return 'evening'

beh['hour_bucket'] = beh['hour_of_day'].apply(hour_to_bucket)
beh['microstate'] = beh['app_name'] + '|' + beh['hour_bucket']

print(f'Behavioral rows: {len(beh):,}')
print(f'\nhour_bucket distribution:')
print(beh['hour_bucket'].value_counts())
print(f'\nUnique microstates (app x hour_bucket): {beh["microstate"].nunique()}')

---

## Section 2: Macrostate Metrics per Participant x Period

In [ ]:
# =============================================================================
# H_joint, dominant config, dominant config share -- per participant x period
# =============================================================================
def shannon_entropy_counts(counts):
    p = counts / counts.sum()
    p = p[p > 0]
    return float(-(p * np.log2(p)).sum())

macro_records = []
period_dists = {}   # (pid, period) -> full microstate probability Series (for JS divergence)

for pid, pgrp in beh.groupby('participant_id_canonical'):
    row = {'participant_id_canonical': pid}
    for period in ['baseline', 'intervention']:
        sub = pgrp[pgrp['study_period'] == period]
        if len(sub) < 5:
            row[f'H_joint_{period}'] = np.nan
            row[f'dominant_config_{period}'] = np.nan
            row[f'dominant_config_share_{period}'] = np.nan
            period_dists[(pid, period)] = None
            continue
        counts = sub['microstate'].value_counts()
        row[f'H_joint_{period}'] = shannon_entropy_counts(counts)
        row[f'dominant_config_{period}'] = counts.index[0]
        row[f'dominant_config_share_{period}'] = float(counts.iloc[0] / counts.sum())
        period_dists[(pid, period)] = counts / counts.sum()
    macro_records.append(row)

macro_df = pd.DataFrame(macro_records)
print('H_joint summary:')
print(macro_df[['H_joint_baseline', 'H_joint_intervention']].describe().round(4))

In [ ]:
# =============================================================================
# Persistence: does the dominant (app, hour) configuration stay the same?
# =============================================================================
macro_df['config_persistent'] = (
    (macro_df['dominant_config_baseline'] == macro_df['dominant_config_intervention'])
    & macro_df['dominant_config_baseline'].notna()
    & macro_df['dominant_config_intervention'].notna()
).astype(int)
macro_df.loc[macro_df['dominant_config_baseline'].isna() | macro_df['dominant_config_intervention'].isna(), 'config_persistent'] = np.nan

n_valid = macro_df['config_persistent'].notna().sum()
n_persistent = macro_df['config_persistent'].sum()
print(f'Config persistent: {int(n_persistent)} / {n_valid} ({100*n_persistent/n_valid:.1f}%)')

In [ ]:
# =============================================================================
# Susceptibility: Jensen-Shannon divergence between baseline and intervention
# microstate distributions, per participant (aligned over the union of microstates)
# =============================================================================
susceptibility_vals = []
for pid in macro_df['participant_id_canonical']:
    p_base = period_dists.get((pid, 'baseline'))
    p_intv = period_dists.get((pid, 'intervention'))
    if p_base is None or p_intv is None:
        susceptibility_vals.append(np.nan)
        continue
    all_states = sorted(set(p_base.index) | set(p_intv.index))
    v_base = np.array([p_base.get(s, 0.0) for s in all_states])
    v_intv = np.array([p_intv.get(s, 0.0) for s in all_states])
    # jensenshannon returns the DISTANCE (sqrt of divergence); square it for the divergence in bits (base=2)
    dist = jensenshannon(v_base, v_intv, base=2)
    susceptibility_vals.append(float(dist ** 2) if not np.isnan(dist) else np.nan)

macro_df['susceptibility'] = susceptibility_vals
print('Susceptibility (JS divergence, bits) summary:')
print(macro_df['susceptibility'].describe().round(4))

---

## Section 3: InT Bridge — Mandatory Sanity Gate (H_joint >= max(H_app, H_hour_bucket))

**Bug found and fixed here (2026-08-06).** First attempt compared `H_joint` (built on the
4-category `hour_bucket`) against InT's `H_hour` -- but InT's `H_hour` is computed on the
FULL 24-value `hour_of_day` (never bucketed there). Comparing a joint entropy built on a
coarsened variable against a marginal entropy built on the fine-grained version of that same
variable is not a valid application of the `H(X,Y) >= H(Y)` identity -- it produced
near-universal "violations" (239/240 baseline, 240/240 intervention) because a 24-category
marginal carries far more raw entropy than a 4-category one, regardless of any joint
structure. Fixed by computing `H_hour_bucket` FRESH, on the exact same `hour_bucket`
partition used to build `H_joint` -- not reusing InT's differently-grained feature.
`H_app` required no such fix (verified InT's `H_app` uses the same full app_name categories
this notebook does -- confirmed by hand for a sample participant, values match to the 4th
decimal).

**Second, smaller bug found immediately after fixing the first:** the gate assertion
initially used `.dropna().all()` on the boolean `gate_valid_*` column, assuming missing-data
participants would show up as NaN there. They don't -- `NaN >= number` evaluates to `False`
in pandas/numpy, not NaN, so `gate_valid_*` had explicit `False` for the ~3 participants
with <5 events per period, and `dropna()` had nothing to drop. The print statement (which
correctly filtered on `H_joint`'s own null mask) showed 240/240 passing while the assertion
below it still fired. Fixed by filtering on `H_joint`'s null mask explicitly before checking
`.all()`, matching what the print logic already did correctly.

In [ ]:
# =============================================================================
# H_hour_bucket computed FRESH, on the same 4-category partition as H_joint (NOT
# InT's 24-value H_hour -- see markdown cell above for why that comparison is invalid).
# =============================================================================
hour_bucket_records = []
for pid, pgrp in beh.groupby('participant_id_canonical'):
    row = {'participant_id_canonical': pid}
    for period in ['baseline', 'intervention']:
        sub = pgrp[pgrp['study_period'] == period]
        row[f'H_hour_bucket_{period}'] = (
            shannon_entropy_counts(sub['hour_bucket'].value_counts()) if len(sub) >= 5 else np.nan
        )
    hour_bucket_records.append(row)
hour_bucket_df = pd.DataFrame(hour_bucket_records)
print('H_hour_bucket (fresh, 4-category) summary:')
print(hour_bucket_df[['H_hour_bucket_baseline', 'H_hour_bucket_intervention']].describe().round(4))

In [ ]:
# =============================================================================
# H_joint >= max(H_app, H_hour_bucket) for every participant, every period -- MUST hold.
# Same discipline that caught 2 real bugs in the InT notebook's NeT-InT bridge, and this
# notebook's own comparability bug above.
# =============================================================================
entropy_df = pd.read_csv(INT_DIR / 'participant_entropy_metrics.csv')
bridge_check = macro_df.merge(
    entropy_df[['participant_id_canonical', 'H_app_baseline', 'H_app_intervention']],
    on='participant_id_canonical', how='left'
).merge(hour_bucket_df, on='participant_id_canonical', how='left')

for period in ['baseline', 'intervention']:
    bridge_check[f'max_marginal_{period}'] = bridge_check[[f'H_app_{period}', f'H_hour_bucket_{period}']].max(axis=1)
    # Audit fix (2026-08-06): np.where so NaN H_joint stores NaN in gate_valid, not
    # False (NaN >= number evaluates False in pandas -- would misrepresent "not
    # applicable" as "gate violated" in the exported CSV, same class of bug already
    # fixed in the InT notebook's net_int_bridge_check.csv).
    bridge_check[f'gate_valid_{period}'] = np.where(
        bridge_check[f'H_joint_{period}'].notna(),
        bridge_check[f'H_joint_{period}'] >= (bridge_check[f'max_marginal_{period}'] - 1e-9),
        np.nan)
    n_valid = bridge_check[f'gate_valid_{period}'].sum()
    n_total = bridge_check[[f'H_joint_{period}', f'max_marginal_{period}']].dropna().shape[0]
    n_fail = bridge_check[bridge_check[f'gate_valid_{period}'] == False][f'H_joint_{period}'].notna().sum()
    print(f'{period}: gate holds for {n_valid}/{n_total} (failures: {n_fail})')

# NOTE: NaN >= number evaluates to False in pandas/numpy (not NaN), so gate_valid_* has
# explicit False (not NaN) for participants with missing H_joint -- must filter on
# H_joint's own null mask before checking .all(), not rely on dropna() on gate_valid itself.
valid_mask_baseline = bridge_check['H_joint_baseline'].notna()
valid_mask_intervention = bridge_check['H_joint_intervention'].notna()
assert bridge_check.loc[valid_mask_baseline, 'gate_valid_baseline'].all(), \
    'StM-InT sanity gate violated (baseline)!'
assert bridge_check.loc[valid_mask_intervention, 'gate_valid_intervention'].all(), \
    'StM-InT sanity gate violated (intervention)!'
print('\nStM-InT sanity gate PASSED: H_joint >= max(H_app, H_hour_bucket) for all valid participants, both periods.')

---

## Section 4: Condition x Period Summary and Psychosocial Correlates

In [ ]:
# =============================================================================
# Condition x period macrostate summary
# =============================================================================
cond_lookup = beh.groupby('participant_id_canonical')['study_condition'].agg(lambda s: s.mode().iloc[0])
macro_df['study_condition'] = macro_df['participant_id_canonical'].map(cond_lookup)

cond_rows = []
for cond in ['control', 'default', 'intervention']:
    sub = macro_df[macro_df['study_condition'] == cond]
    cond_rows.append(dict(
        study_condition=cond, n=len(sub),
        H_joint_baseline_mean=sub['H_joint_baseline'].mean(),
        H_joint_intervention_mean=sub['H_joint_intervention'].mean(),
        pct_persistent=100*sub['config_persistent'].mean(),
        susceptibility_mean=sub['susceptibility'].mean(),
    ))
condition_macrostate_summary = pd.DataFrame(cond_rows).round(4)
print(condition_macrostate_summary.to_string(index=False))

f_stat, p_anova = stats.f_oneway(*[
    macro_df.loc[macro_df.study_condition==c, 'susceptibility'].dropna()
    for c in ['control','default','intervention']
])
print(f'\nANOVA susceptibility ~ condition: F={f_stat:.3f}, p={p_anova:.4f}')

In [ ]:
# =============================================================================
# Psychosocial correlations (exploratory): susceptibility vs addiction/self_control
# =============================================================================
survey = pd.read_csv(FEAT / 'ds6_survey_features.csv')
merged_psych = macro_df.merge(
    survey[['participant_id_canonical', 'addiction_index_pre', 'self_control_index_pre']],
    on='participant_id_canonical', how='left'
)
corr_records = []
for col in ['H_joint_baseline', 'H_joint_intervention', 'susceptibility']:
    for pcol in ['addiction_index_pre', 'self_control_index_pre']:
        sub = merged_psych[[col, pcol]].dropna()
        if len(sub) >= 10:
            r, p = stats.pearsonr(sub[col], sub[pcol])
            corr_records.append(dict(macrostate_measure=col, psychosocial=pcol, n=len(sub), r=round(r,4), p=round(p,4)))
susceptibility_psychosocial_correlations = pd.DataFrame(corr_records)
print(susceptibility_psychosocial_correlations.sort_values('p').to_string(index=False))

In [ ]:
# =============================================================================
# Figure 1: H_joint by period
# =============================================================================
fig, ax = plt.subplots(figsize=(7, 5))
data = [macro_df['H_joint_baseline'].dropna(), macro_df['H_joint_intervention'].dropna()]
ax.violinplot(data, showmeans=True)
ax.set_xticks([1,2]); ax.set_xticklabels(['baseline', 'intervention'])
ax.set_ylabel('H_joint (bits)')
ax.set_title('DS6 StM -- joint (app, hour) configuration entropy by period')
plt.tight_layout()
fig.savefig(FIGURES / 'fig1_stm_hjoint_by_period.png', dpi=150)
plt.show()
print('Saved fig1_stm_hjoint_by_period.png')

In [ ]:
# =============================================================================
# Figure 2: persistence distribution (by condition)
# =============================================================================
fig, ax = plt.subplots(figsize=(7, 5))
pct = condition_macrostate_summary.set_index('study_condition')['pct_persistent']
ax.bar(pct.index, pct.values, color='#55A868')
ax.set_ylabel('% participants with persistent dominant configuration')
ax.set_title('DS6 StM -- configuration persistence by condition')
ax.set_ylim(0, 100)
plt.tight_layout()
fig.savefig(FIGURES / 'fig2_stm_persistence_distribution.png', dpi=150)
plt.show()
print('Saved fig2_stm_persistence_distribution.png')
print(f'Overall persistence: {100*macro_df["config_persistent"].mean():.1f}%')

In [ ]:
# =============================================================================
# Figure 3: susceptibility distribution
# =============================================================================
fig, ax = plt.subplots(figsize=(7, 5))
ax.hist(macro_df['susceptibility'].dropna(), bins=40, color='#C44E52', alpha=0.8)
ax.axvline(macro_df['susceptibility'].mean(), color='black', linestyle='--',
           label=f'mean={macro_df["susceptibility"].mean():.3f}')
ax.set_xlabel('susceptibility (JS divergence, bits)')
ax.set_ylabel('Participants')
ax.set_title('DS6 StM -- distributional shift (baseline -> intervention)')
ax.legend()
plt.tight_layout()
fig.savefig(FIGURES / 'fig3_stm_susceptibility_distribution.png', dpi=150)
plt.show()
print('Saved fig3_stm_susceptibility_distribution.png')

In [ ]:
# =============================================================================
# Figure 4: InT bridge scatter (H_joint vs max marginal)
# =============================================================================
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, period in zip(axes, ['baseline', 'intervention']):
    sub = bridge_check[[f'H_joint_{period}', f'max_marginal_{period}']].dropna()
    ax.scatter(sub[f'max_marginal_{period}'], sub[f'H_joint_{period}'], alpha=0.5, s=20)
    lims = [0, max(sub.max())]
    ax.plot(lims, lims, color='gray', linewidth=0.8, linestyle='--', label='H_joint = max(H_app,H_hour)')
    ax.set_xlabel('max(H_app, H_hour) (InT, notebook 75)')
    ax.set_ylabel('H_joint (this notebook)')
    ax.set_title(f'{period}: sanity gate (points on/above line = valid)')
    ax.legend(fontsize=8)
plt.tight_layout()
fig.savefig(FIGURES / 'fig4_stm_int_bridge_scatter.png', dpi=150)
plt.show()
print('Saved fig4_stm_int_bridge_scatter.png')

In [ ]:
# =============================================================================
# Export all outputs
# =============================================================================
out_cols = ['participant_id_canonical', 'study_condition',
            'H_joint_baseline', 'dominant_config_baseline', 'dominant_config_share_baseline',
            'H_joint_intervention', 'dominant_config_intervention', 'dominant_config_share_intervention',
            'config_persistent', 'susceptibility']
participant_macrostate_metrics = macro_df[out_cols].copy()
participant_macrostate_metrics.to_csv(REPORTS / 'participant_macrostate_metrics.csv', index=False)
print(f'Saved participant_macrostate_metrics.csv ({len(participant_macrostate_metrics)} rows)')

int_bridge_check = bridge_check[['participant_id_canonical',
    'H_joint_baseline', 'max_marginal_baseline', 'gate_valid_baseline',
    'H_joint_intervention', 'max_marginal_intervention', 'gate_valid_intervention']].copy()
int_bridge_check.to_csv(REPORTS / 'int_bridge_check.csv', index=False)
print(f'Saved int_bridge_check.csv ({len(int_bridge_check)} rows)')

condition_macrostate_summary.to_csv(REPORTS / 'condition_macrostate_summary.csv', index=False)
print(f'Saved condition_macrostate_summary.csv ({len(condition_macrostate_summary)} rows)')

susceptibility_psychosocial_correlations.to_csv(REPORTS / 'susceptibility_psychosocial_correlations.csv', index=False)
print(f'Saved susceptibility_psychosocial_correlations.csv ({len(susceptibility_psychosocial_correlations)} rows)')

In [ ]:
# =============================================================================
# model_run_summary.md
# =============================================================================
summary_md = f"""# DS6 StM -- Model Run Summary

**Run at:** {MODEL_RUN_TS}
**Notebook:** notebooks/71_ds6_statistical_mechanics.ipynb
**Blueprint:** outputs/reports/stage4/step1_modeling/ds6_stm_blueprint.md

## Key results
- Mean H_joint (baseline / intervention): {macro_df['H_joint_baseline'].mean():.4f} / {macro_df['H_joint_intervention'].mean():.4f} bits
- Configuration persistence (dominant (app,hour) same both periods): {100*macro_df['config_persistent'].mean():.1f}% ({int(macro_df['config_persistent'].sum())}/{int(macro_df['config_persistent'].notna().sum())})
- Mean susceptibility (JS divergence): {macro_df['susceptibility'].mean():.4f} bits
- StM-InT sanity gate: PASSED (H_joint >= max(H_app, H_hour) for all valid participants, both periods)
- ANOVA susceptibility ~ condition: F={f_stat:.3f}, p={p_anova:.4f}

## Proxy warning
No literal energy/temperature/equilibrium quantities are computed (FLAG StM-1) -- DS6's
unbounded StoP state variable makes DS5's beta=2/sigma^2 bridge inapplicable even as a loose
analogy. H_joint, persistence, and susceptibility are standard information-theoretic /
distributional-shift quantities, describing patterns in observed (app, time) configurations,
not directly observed psychological states (habit strength, friction sensitivity).

## Known limitations
- hour_bucket is a 4-category coarsening of hour_of_day (paper names 'time of day' as a
  configuration property, not a specific granularity) -- results are sensitive to bucket
  choice in principle, not tested here.
- Persistence is based on a single dominant configuration per period -- a participant whose
  distribution is genuinely diffuse (no strong mode) can show a persistence result driven by
  a small vote margin; `dominant_config_share` should be checked alongside `config_persistent`.
- Psychosocial correlations exploratory, uncorrected ({len(susceptibility_psychosocial_correlations)} tests).
- study_condition is quasi-experimental (cohort-level), not individually randomized.
"""

(REPORTS / 'model_run_summary.md').write_text(summary_md)
print(summary_md)

---

## Section 5: Interpretation and Limits

### Headline finding: friction conditions show significantly higher susceptibility than control

Control mean susceptibility=0.028 vs default=0.070 (p=0.0011) and vs intervention=0.052
(p=0.0075); default vs intervention not significantly different (p=0.22). DS6's first
clearly CONDITION-differentiated result (not just period-differentiated) across all of
Step 1 -- control participants reorganize their (app, time-of-day) routine significantly
less than either friction arm. Matches the paper's own StM framing directly ("friction-
sensitive user may show a large shift... after a relatively small intervention") and
DS5's own established friction-disrupts-regularity finding.

### What the StM model reveals

| Result | Interpretation |
|---|---|
| H_joint (baseline vs intervention) | Diversity of the (app, time-of-day) configuration space -- distinct from InT's marginal H_app/H_hour |
| Configuration persistence | Whether a participant's single most common (app, hour) routine survives the transition into the intervention period |
| Susceptibility | Magnitude of overall distributional reorganization across the baseline-intervention boundary -- a population-shift analog of the paper's "friction-sensitive" phenotype |

### What this model does NOT show

- Literal thermodynamic quantities -- deliberately not built (FLAG StM-1).
- A causal account of why configurations shift.
- Friction sensitivity at the level of a single event (paper's Example A uses a single
  Instagram-attempt vignette) -- this notebook operates at the aggregated per-period
  distribution level only.

### Forward connections (informational only -- not implemented here)

- **StM -> Step 2 phenotyping:** H_joint, persistence, and susceptibility (all per
  participant) are candidate phenotype dimensions, completing the set from all 5 already-
  complete DS6 families (StoP, NeT, InT, DyS, SyD) plus this one.
- **This closes DS6 Step 1 for all active model families** (StoP, NeT, InT, DyS, SyD, StM).
  GaT and ABM remain formally deferred pending Step 2, matching the DS5 and Structured
  precedent exactly.